# 🎭 AI Avatar All-in-One - Làm Tất Cả Trên Google Colab

> **Bỏ video/ảnh vào → Colab lo hết → Ra kết quả live. Không cần GPU máy local!**

## ⚡ Cách Dùng (3 bước)

1. Upload video hoặc ảnh vào Google Drive folder `AI_Face_Data/input/`
2. **Bật GPU:** Runtime → Change runtime type → **T4 GPU**
3. Chạy từng cell: `Ctrl+Enter` hoặc Runtime → Run all

## 📋 Pipeline Tự Động

```
Video/Ảnh Input → Auto phát hiện mặt → Phân loại góc → Trích landmarks
→ Delaunay Face Warp → Tạo animation → Live demo webcam → Xuất video
```


## 1. Mount Drive & Cài Đặt


In [ ]:
# ============================================================
# 2. FULL PIPELINE: Auto-train + Face Warp + Video + Export
# ============================================================
from pathlib import Path
from datetime import datetime
import cv2, json, os, numpy as np, pickle, time
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
from tqdm import tqdm

# Paths
INPUT_DIR = '/content/drive/MyDrive/AI_Face_Data/input'
OUTPUT_DIR = '/content/drive/MyDrive/AI_Face_Data/output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Trained log
trained_log = {}
log_path = os.path.join(OUTPUT_DIR, '..', 'trained_log.json')
if os.path.exists(log_path):
    with open(log_path) as f: trained_log = json.load(f)

# Session
SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
session_output = f'{OUTPUT_DIR}/{SESSION_ID}'
os.makedirs(session_output, exist_ok=True)

# === SCAN FILES ===
all_files = []
exts = ['*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV',
        '*.jpg','*.JPG','*.jpeg','*.JPEG','*.png','*.PNG']
for ext in exts:
    all_files.extend(Path(INPUT_DIR).glob(ext))
for d in Path(INPUT_DIR).iterdir():
    if d.is_dir():
        for ext in exts:
            all_files.extend(d.glob(ext))

new_files = [f for f in all_files if f.name not in trained_log]
print(f"Files: {len(all_files)} | New: {len(new_files)} | Session: {SESSION_ID}")
for f in new_files: print(f"  [NEW] {f.name}")

if len(new_files) == 0:
    print("[OK] No new files. Done.")
else:
    # === FACE DETECTION ===
    import face_recognition
    def get_landmarks_and_pose(img_rgb):
        h, w = img_rgb.shape[:2]
        locs = face_recognition.face_locations(img_rgb, model='hog')
        if not locs: return None, None, None
        lms = face_recognition.face_landmarks(img_rgb, locs)
        if not lms: return None, None, None
        pts = []
        for f in ['chin','left_eyebrow','right_eyebrow','nose_bridge','nose_tip','left_eye','right_eye','top_lip','bottom_lip']:
            pts.extend(lms[0][f])
        lm = np.array(pts, dtype=np.float32)
        t,r,b,l = locs[0]
        cx = (l+r)/2
        yaw = (cx - w/2)/(w/2)*65
        if (r-l)/max(b-t,1) < 0.5: yaw *= 1.3
        return lm, yaw, img_rgb

    best_frames = {'left':None,'center':None,'right':None}
    best_info = {'left':(0,0,0,None),'center':(0,0,0,None),'right':(0,0,0,None)}
    all_motion = []

    def process_frame(img, idx, ts=0):
        lm, yaw, _ = get_landmarks_and_pose(img)
        if lm is None: return
        q = cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        angle = 'left' if yaw<-12 else ('right' if yaw>12 else 'center')
        all_motion.append((ts, yaw))
        _, _, bq, _ = best_info[angle]
        if bq is None or q > bq:
            best_frames[angle] = img.copy()
            best_info[angle] = (yaw, q, q, lm)

    for f in new_files:
        fn = str(f)
        if f.suffix.lower() in ['.mp4','.avi','.mov']:
            cap = cv2.VideoCapture(fn)
            if not cap.isOpened(): print(f"[SKIP] {f.name}"); continue
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            idx = 0
            while True:
                ret, frame = cap.read()
                if not ret: break
                if idx % 3 == 0:
                    process_frame(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), idx, idx/fps)
                idx += 1
            cap.release()
            print(f"[VIDEO] {f.name}: {idx} frames, {len(all_motion)} detections")
        else:
            img = cv2.imread(fn)
            if img is None: print(f"[SKIP] {f.name}"); continue
            process_frame(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 0, 0)
            print(f"[IMAGE] {f.name}")

    print(f"Motion: {len(all_motion)}")
    missing = []
    for a in ['left','center','right']:
        if best_frames[a] is not None:
            y, q, _, _ = best_info[a]
            print(f"  {a}: yaw={y:.0f}")
        else:
            print(f"  {a}: NOT FOUND")
            missing.append(a)

    if len(missing) < 3:
        # === FACE WARP ===
        img_left, img_center, img_right = best_frames['left'], best_frames['center'], best_frames['right']
        lm_left, lm_center, lm_right = best_info['left'][3], best_info['center'][3], best_info['right'][3]
        yaw_left, yaw_center, yaw_right = float(best_info['left'][0]), float(best_info['center'][0]), float(best_info['right'][0])
        h, w = img_center.shape[:2]

        hull_idx = cv2.convexHull(lm_center.astype(np.int32), returnPoints=False).squeeze()
        border = np.array([[0,0],[w//2,0],[w-1,0],[0,h//2],[w-1,h//2],[0,h-1],[w//2,h-1],[w-1,h-1]], dtype=np.float32)
        all_c = np.vstack([lm_center[hull_idx].astype(np.float32), border])
        all_l = np.vstack([lm_left[hull_idx].astype(np.float32), border])
        all_r = np.vstack([lm_right[hull_idx].astype(np.float32), border])
        tri = Delaunay(all_c)

        def warp_tri(src, dst, s_tri, d_tri):
            sr = cv2.boundingRect(s_tri.astype(np.int32)); dr = cv2.boundingRect(d_tri.astype(np.int32))
            sc = src[sr[1]:sr[1]+sr[3], sr[0]:sr[0]+sr[2]]
            if sc.size == 0: return dst
            M = cv2.getAffineTransform((s_tri-sr[:2]).astype(np.float32), (d_tri-dr[:2]).astype(np.float32))
            dc = cv2.warpAffine(sc, M, (dr[2],dr[3]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            mask = np.zeros((dr[3],dr[2]), dtype=np.float32); cv2.fillConvexPoly(mask, (d_tri-dr[:2]).astype(np.int32), 1.0)
            roi = dst[dr[1]:dr[1]+dr[3], dr[0]:dr[0]+dr[2]]
            if roi.shape == dc.shape:
                roi[:] = (dc * np.stack([mask]*3,-1) + roi * (1-np.stack([mask]*3,-1))).astype(np.uint8)
            return dst

        def morph(src_img, src_all, dst_all):
            res = np.zeros_like(src_img)
            for s in tri.simplices: res = warp_tri(src_img, res, src_all[s], dst_all[s])
            return res

        def rotate_to(target_yaw):
            target_yaw = np.clip(target_yaw, yaw_left, yaw_right)
            if abs(target_yaw - yaw_center) < 1: return img_center.copy()
            if target_yaw <= yaw_center:
                t = (target_yaw - yaw_left) / (yaw_center - yaw_left + 1e-8)
                return morph(img_left, all_l, (1-t)*all_l + t*all_c)
            else:
                t = (target_yaw - yaw_center) / (yaw_right - yaw_center + 1e-8)
                return morph(img_center, all_c, (1-t)*all_c + t*all_r)

        # Show test
        tests = [yaw_left, (yaw_left+yaw_center)/2, yaw_center, (yaw_center+yaw_right)/2, yaw_right]
        fig, axes = plt.subplots(1,5,figsize=(15,3))
        for i, a in enumerate(tests):
            axes[i].imshow(rotate_to(a))
            axes[i].set_title(f'yaw={a:.0f}'); axes[i].axis('off')
        plt.tight_layout(); plt.show()
        print("[OK] Face Warp ready!")

        # === CREATE VIDEO ===
        if len(all_motion) > 10:
            timestamps, yaws = zip(*all_motion); print(f"Real motion: {len(yaws)} pts")
        else:
            f_seg = 60
            yl = list(np.linspace(yaw_left, yaw_center, f_seg))
            yr = list(np.linspace(yaw_center, yaw_right, f_seg))
            yaws = yl + yr[1:] + list(reversed(yl))[1:]
            print(f"Generated motion: {len(yaws)} frames")

        out_path = f'{session_output}/avatar_animation.mp4'
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(out_path, fourcc, 30, (w, h))
        for target_yaw in tqdm(yaws[:300], desc="Render"):
            r = rotate_to(target_yaw)
            cv2.putText(r, f"Yaw: {target_yaw:+.0f}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            out.write(cv2.cvtColor(r, cv2.COLOR_RGB2BGR))
        out.release()
        print(f"[OK] Video: {out_path}")

        # === EXPORT ===
        face_data = {
            'images': {'left':img_left,'center':img_center,'right':img_right},
            'landmarks': {'left':lm_left,'center':lm_center,'right':lm_right},
            'yaws': {'left':yaw_left,'center':yaw_center,'right':yaw_right},
            'motion': all_motion,
        }
        for a in ['left','center','right']:
            if best_frames[a] is not None:
                cv2.imwrite(f'{session_output}/best_{a}.jpg', cv2.cvtColor(best_frames[a], cv2.COLOR_RGB2BGR))
        with open(f'{session_output}/face_data.pkl', 'wb') as f: pickle.dump(face_data, f)
        session_log = {'session_id': SESSION_ID, 'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                       'files': {f.name:{} for f in new_files}}
        with open(f'{session_output}/trained_log.json', 'w') as fl: json.dump(session_log, fl, indent=2)
        for f in new_files: trained_log[f.name] = {'session': SESSION_ID}
        print(f"[OK] Saved to {session_output}/")
    else:
        print(f"[SKIP] Thieu goc: {missing}")


SyntaxError: incomplete input (1781229187.py, line 9)

## 8. Tổng Kết

✅ **Pipeline Hoàn Thành!**

📥 **Kết quả lưu tại:** `AI_Face_Data/output/[SESSION_ID]/`

- `avatar_animation.mp4` — Video animation
- `best_left/center/right.jpg` — Ảnh tốt nhất mỗi góc
- `face_data.pkl` — Model cho máy local

📋 **Nhật ký:** `AI_Face_Data/trained_log.json` — tự ghi file nào đã train

🔁 **Lần sau có video mới:**

1. Upload vào `AI_Face_Data/input/`
2. Restart & Run All → tự bỏ qua video cũ, chỉ train video mới
3. Kết quả ra folder riêng: `output/20260727_143000/`

🗑️ **Muốn train lại video cũ:** Xóa tên file trong `trained_log.json` hoặc đổi tên video
